        # 02 - Building a plant

        [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pbtamarona/openpytea-workshop/blob/main/notebooks/02_building_a_plant.ipynb)

        A `Plant` combines an equipment list with economic assumptions and returns everything a TEA report needs:
fixed capital investment, fixed and variable operating costs, revenue, yearly cash flows, and the profitability
metrics (NPV, IRR, ROI, payback time, levelized cost). In this notebook we build the green-ammonia plant step by
step and learn how to read and change its assumptions.

Expected time: 35 minutes.

        ## Learning goals

        - Write a plant configuration dictionary with financial assumptions, products, and variable OPEX inputs.
- Compute and read the fixed capital investment (ISBL, OSBL, design and engineering, contingency, location factor).
- Compute fixed OPEX, variable OPEX, revenue, and the yearly cash flow.
- Interpret NPV, IRR, ROI, payback time, and the levelized cost of product, and know which assumptions each depends on.
- Change assumptions with `update_configuration()` and re-run the calculation.

        ## Background references

        - Towler, G., & Sinnott, R. (2022). *Chemical Engineering Design* (3rd ed.). Elsevier. https://doi.org/10.1016/C2019-0-02025-0 (chapters 7 to 9)
- Turton, R., Shaeiwitz, J.A., Bhattacharyya, D., & Whiting, W.B. (2018). *Analysis, Synthesis, and Design of Chemical Processes* (5th ed.). Prentice Hall. (chapters 7 to 10)
- OpenPyTEA documentation: https://openpytea.readthedocs.io

## 0) Setup: rebuild the equipment list from notebook 01

In [ ]:
# Run this cell first. It installs OpenPyTEA if it is missing (for example on Google Colab).
try:
    import openpytea
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "openpytea"])
    import openpytea

print("OpenPyTEA version:", openpytea.__version__)

In [ ]:
from openpytea import Equipment

# Equipment of the green-ammonia synthesis loop (sizes from the process simulation, see notebook 01)
compressor  = Equipment(name="C-101 Syngas compressor", param=1_200, process_type="Fluids",
                        category="Compressors, fans, & blowers", type="Compressor, centrifugal",
                        material="Carbon steel")
motor       = Equipment(name="M-101 Compressor motor", param=1_200 / 0.90, process_type="Electrical",
                        category="Motors & generators", type="Totally enclosed motor")
exchanger   = Equipment(name="E-101 Feed/effluent exchanger", param=250, process_type="Fluids",
                        category="Heat Exchangers", type="U-tube shell & tube",
                        material="304 stainless steel")
converter   = Equipment(name="R-101 Ammonia converter", param=12_000, process_type="Fluids",
                        category="Reactors", type="Tubular fixed bed", material="304 stainless steel")
separator   = Equipment(name="V-101 Ammonia separator", param=25, process_type="Fluids",
                        category="Pressure vessels", type="Vertical", material="Carbon steel")
chiller     = Equipment(name="K-101 Refrigeration unit", param=800, process_type="Fluids",
                        category="Utilities", type="Packaged mechanical refrigerator")
product_pump = Equipment(name="P-101 Product pump", param=20, process_type="Fluids",
                         category="Pumps", type="Single-stage centrifugal", material="Carbon steel")

equipment_list = [compressor, motor, exchanger, converter, separator, chiller, product_pump]

print(f"ISBL (sum of direct costs): {sum(e.direct_cost for e in equipment_list):,.0f} USD")

## 1) The plant configuration

A `Plant` is created from one dictionary. Only `plant_name`, `process_type`, `equipment`, and `plant_products`
are strictly required; everything else has a documented default. Units are up to you, but they must be
consistent: OpenPyTEA multiplies **daily** quantities by 365 and by the plant utilization to get yearly values,
and prices are per unit of that quantity.

In [ ]:
from openpytea import Plant

ammonia_config = {
    "plant_name": "Green Ammonia",
    "process_type": "Fluids",            # "Solids" | "Fluids" | "Mixed" -> sets default OSBL, D&E, contingency factors
    "country": "Netherlands",            # location factor lookup (see Plant.locFactors); "region" is needed for some countries
    "equipment": equipment_list,

    # Financial assumptions
    "interest_rate": 0.08,               # discount rate
    "project_lifetime": 20,              # years, including construction
    "plant_utilization": 0.90,           # fraction of the year on stream
    "tax_rate": 0.25,

    # Products: the FIRST entry is the main product used for the levelized cost
    "plant_products": {
        "ammonia": {"production": 100_000, "price": 0.95},   # kg/day, USD/kg
    },

    # Variable OPEX: daily consumption and unit price of every feed and utility
    "variable_opex_inputs": {
        "hydrogen":    {"consumption": 17_800, "price": 2.00},   # kg/day,  USD/kg
        "nitrogen":    {"consumption": 82_400, "price": 0.05},   # kg/day,  USD/kg
        "electricity": {"consumption": 60_000, "price": 0.08},   # kWh/day, USD/kWh
    },
}

ammonia_plant = Plant(ammonia_config)

## 2) Fixed capital investment

$$\text{FCI} = \text{ISBL} \cdot (1 + OS) \cdot (1 + D\&E + X) \cdot LF$$

ISBL is the sum of the equipment direct costs. The offsite (OS), design and engineering (D&E), and contingency
(X) factors default to 0.3, 0.3, and 0.1 for fluid plants (Towler & Sinnott). The location factor LF comes from
the country (and region). Any of these can be overridden.

In [ ]:
ammonia_plant.calculate_fixed_capital(print_results=True)

In [ ]:
# The location factors OpenPyTEA knows about
Plant.locFactors

## 3) Operating costs and revenue

- **Variable OPEX**: consumption × price × 365 × utilization, for each item in `variable_opex_inputs`.
- **Fixed OPEX**: operating labor (from an operator count based on the number of process steps, or a number you
  give), supervision, overheads, maintenance (5 % of ISBL), taxes and insurance, and so on, following Turton et al.
- **Revenue**: production × price × 365 × utilization, for each product.

In [ ]:
ammonia_plant.calculate_variable_opex(print_results=True)

In [ ]:
ammonia_plant.calculate_fixed_opex(print_results=True)
print(f"Operators hired (auto-estimated): {ammonia_plant.calculate_operators_hired()}")

In [ ]:
ammonia_plant.calculate_revenue(print_results=True)

## 4) Cash flow and profitability metrics

`calculate_cash_flow()` builds the year-by-year table. By default the capital is spent over three construction
years (30 / 60 / 10 %), production ramps up (0 / 0 / 40 / 80 / 100 %), depreciation follows MACRS, and taxes are
paid with a one-year lag. Working capital (15 % of FCI by default) is drawn at start-up and released at the end.

In [ ]:
ammonia_plant.calculate_cash_flow(print_results=True)

Once the cash flows exist, the metrics follow:

| Metric | Meaning | Method |
|---|---|---|
| NPV | Present value of all cash flows at the interest rate | `calculate_npv()` |
| LCOP | Levelized cost of product: the selling price at which NPV = 0, after crediting by-products | `calculate_levelized_cost()` |
| PBT | Payback time: FCI divided by average annual cash flow | `calculate_payback_time()` |
| ROI | Average yearly net profit over total investment | `calculate_roi()` |
| IRR | Discount rate at which NPV = 0 | `calculate_irr()` |

`calculate_all()` runs every step above in the right order.

In [ ]:
ammonia_plant.calculate_all()

print(f"Fixed capital investment : {ammonia_plant.fixed_capital:>14,.0f} USD")
print(f"Variable OPEX            : {ammonia_plant.variable_production_costs:>14,.0f} USD/yr")
print(f"Fixed OPEX               : {ammonia_plant.fixed_production_costs:>14,.0f} USD/yr")
print(f"Revenue                  : {ammonia_plant.revenue:>14,.0f} USD/yr")
print(f"Levelized cost of ammonia: {ammonia_plant.levelized_cost:>14.3f} USD/kg")
print(f"Net present value        : {ammonia_plant.npv:>14,.0f} USD")
print(f"Internal rate of return  : {ammonia_plant.irr:>14.1%}")
print(f"Return on investment     : {ammonia_plant.roi:>14.1f} %")
print(f"Payback time             : {ammonia_plant.payback_time:>14.1f} years")

> **Reading the result.** The levelized cost (about 0.89 USD/kg) sits just below the assumed selling price of
> 0.95 USD/kg, so the project is marginally profitable: the NPV is a few million USD and the IRR is a little
> above the 8 % discount rate. Hydrogen is by far the largest cost item. Keep these numbers in mind: the next
> notebooks ask how robust this conclusion is.

## 5) Changing assumptions

`update_configuration()` merges a partial dictionary into the existing configuration (nested dictionaries are
merged, not replaced), so you can change one number and recalculate. Overrides exist for almost every default:
`fixed_capital_factors`, `fixed_opex_factors`, `loc_factor`, `operators_hired`, `capex_ramp`,
`production_ramp`, `depreciation`, `working_capital`, `additional_capex_cost` and `additional_capex_years`.

In [ ]:
from copy import deepcopy

variant = deepcopy(ammonia_plant)
variant.update_configuration({
    "plant_name": "Green Ammonia (cheaper H2)",
    "variable_opex_inputs": {"hydrogen": {"price": 1.50}},       # only the price changes; consumption is kept
    "fixed_opex_factors": {"maintenance": 0.03},                 # 3 % of ISBL instead of 5 %
})
variant.calculate_all()

print(f"LCOP: {ammonia_plant.levelized_cost:.3f} -> {variant.levelized_cost:.3f} USD/kg")
print(f"NPV : {ammonia_plant.npv:,.0f} -> {variant.npv:,.0f} USD")

## 6) Exercise 3: where should the plant be built?

Location factors scale the ISBL to local construction costs. Compare the fixed capital investment and the
levelized cost of the ammonia plant for these locations:

- Netherlands
- United States, region `"Gulf Coast"`
- China, region `"indigenous"` (locally sourced equipment)
- India

Use `deepcopy(ammonia_plant)` and `update_configuration({"country": ..., "region": ...})` in a loop and collect
the results in a pandas DataFrame. Note that a country with regional factors requires `region`, and one without
should have `"region": None`.

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
import pandas as pd
from copy import deepcopy

locations = [("Netherlands", None), ("United States", "Gulf Coast"), ("China", "indigenous"), ("India", None)]
rows = []
for country, region in locations:
    p = deepcopy(ammonia_plant)
    p.update_configuration({"country": country, "region": region})
    p.calculate_all()
    rows.append({"location": f"{country} {region or ''}".strip(), "FCI [MUSD]": p.fixed_capital / 1e6,
                 "LCOP [USD/kg]": p.levelized_cost, "NPV [MUSD]": p.npv / 1e6})
pd.DataFrame(rows).set_index("location").round(3)
```
</details>

## 7) Exercise 4: the break-even price and the effect of catalyst replacement

1. By definition the levelized cost is the price at which NPV = 0 **before taxes**. Check it: set the ammonia
   price equal to the computed `levelized_cost` on a copy of the plant and print the NPV. It comes out negative,
   because the 25 % tax is still paid on the profitable years. Now also set `"tax_rate": 0` and confirm that the
   NPV becomes zero.
2. The converter catalyst must be replaced every 5 years at 1.2 MUSD each time. Add this with
   `"additional_capex_cost": [1.2e6, 1.2e6, 1.2e6]` and `"additional_capex_years": [7, 12, 17]`, run
   `calculate_all(additional_capex=True)` and report how much the NPV and levelized cost change.

In [ ]:
# YOUR CODE HERE

<details>
<summary><b>Solution</b> (click to expand)</summary>

```python
breakeven = deepcopy(ammonia_plant)
breakeven.update_configuration({"plant_products": {"ammonia": {"price": ammonia_plant.levelized_cost}}})
breakeven.calculate_all()
print(f"NPV at price = LCOP, 25 % tax: {breakeven.npv:,.0f} USD")

breakeven.update_configuration({"tax_rate": 0.0})
breakeven.calculate_all()
print(f"NPV at price = LCOP, no tax  : {breakeven.npv:,.0f} USD")

with_catalyst = deepcopy(ammonia_plant)
with_catalyst.update_configuration({"additional_capex_cost": [1.2e6, 1.2e6, 1.2e6],
                                    "additional_capex_years": [7, 12, 17]})
with_catalyst.calculate_all(additional_capex=True)
print(f"LCOP: {ammonia_plant.levelized_cost:.3f} -> {with_catalyst.levelized_cost:.3f} USD/kg")
print(f"NPV : {ammonia_plant.npv:,.0f} -> {with_catalyst.npv:,.0f} USD")
```
</details>

## Recap

- One dictionary configures a `Plant`; the first product is the main product used for the levelized cost.
- FCI = ISBL × (1 + OS) × (1 + D&E + X) × LF, all factors visible and overridable.
- `calculate_all()` gives FCI, OPEX, revenue, cash flow, NPV, LCOP, PBT, ROI, IRR in one call.
- `deepcopy` + `update_configuration()` is the pattern for scenarios; we automate it in the next notebook.

**Next:** [03 - Cost breakdowns and sensitivity](03_cost_breakdowns_and_sensitivity.ipynb)